# Matrixize `rMATS` Data

## Purpose: 

Convert `rMATS` results files into matrices

## Packages and Options

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import glob, os 

## Literals

In [2]:
cell_lines = ["HepG2", "K562"]
splice_types = ["A3SS", "A5SS", "SE", "MXE", "RI"]

# directory name to pull either batch-corrected or not batch-corrected files 
norm_or_not_pattern = "MATS_output"

# columns that have counts info in all rMATS files 
count_columns = ["IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2"]

# choose whether you want skipping and junction counts to be summated or not
get_total_counts = True 

# order of samples if junction and skipping counts are SUMMATED 
summated_sample_ordering = ["KO_Sample_1", "KO_Sample_2", "CTRL_Sample_1", "CTRL_Sample_2"]
# order of samples if junction and skipping counts are NOT SUMMATED
non_summated_sample_ordering = ["KO_Sample_1_IJC", "KO_Sample_2_IJC", "KO_Sample_1_SJC", "KO_Sample_2_SJC", "CTRL_Sample_1_IJC", "CTRL_Sample_2_IJC", "CTRL_Sample_1_SJC", "CTRL_Sample_2_SJC"]



## Matrixization Algorithm

In [3]:
def matrixize_rmats_table(file = None, get_total_counts=None):
    """
    Takes all rMATS files and extracts genomic features from them to create a matrix using junction and skipping counts. 
    
    Parameters:
        files (list): list of file paths where the rMATS files can be found (default = None)
        get_total_counts (bool): whether to sum junction and skipping counts or keep them separate (default = None)         
    
    Returns: 
        matrix (pandas.DataFrame): a Pandas DataFrame where features are columns and rows are samples
        
    """
    
    assert file!=None and get_total_counts!=None
    
     # need to extract rbp and cell line from path 
    rbp_cell_line = None

    for path_part in file.split("/"): 
        # save the entire line that has the rbp-batch-cell_line nomenclature
        if "HepG2" in path_part or "K562" in path_part: 
            rbp_cell_line = path_part 

    assert rbp_cell_line != None, rbp_cell_line
    
    # dictionary where genomic position feature is key and value is sub-dict 
    # sub-dict has key for sample and value of that is the counts 
    # NOTE: if "get_total_counts=False", you will have double the sampls since "SJC" and "IJC" will be kept separate. 
    matrix_dict = {}

    # load as dataframe 
    tmp_df = pd.read_csv(file,sep="\t")

    # get index position of concatenation start string 
    # we are making strings from the exact chromosome, strand and genomic positions 
    # to do so, we need to concatenate all rows after "chr" column until the "ID.1" column 
    # this is a known assumption that all the chromosome, strand, and genomic position information 
    # is between the "chr" column upto and excluding "ID.1" column
    concatenation_start = tmp_df.columns.tolist().index("chr")
    concatenation_end = tmp_df.columns.tolist().index("ID.1")

    # take all columns to make feature name 
    # convert them to strings 
    # concatenate each column row-wise with "_" as delimiter
    # (e.g. chr17_-_62496792_62497000_62496792_62496891_62498127_62498187)
    tmp_df["Feature"] = tmp_df.iloc[
        :,concatenation_start:concatenation_end
    ].astype("str").apply(
        lambda x: '_'.join(x.values.tolist()), axis=1
    )

    # take all the counts that are comma separated per column 
    # and combine them into one column that is entirely comma-separated 
    # e.g. 780,750	758,759	260,253	543,571	 becomes 780,750,758,759,260,253,543,571
    tmp_df["Counts"] = tmp_df[count_columns].astype(str).apply(
        lambda x: ",".join(x.values.tolist()),axis=1
    )

    # subset to the columns involving features and counts 
    tmp_df = tmp_df[["Feature", "Counts"]]

    # for each feature we are now going to sum up the skipping and inclusion junction counts
    # there are 8 numbers corresponding to 4 samples
    # we are using the sample count summation indices dictionary that tells us which indices to pull from list
    for row in tmp_df.itertuples():

        feature = row[1]
        matrix_dict[feature] = {}
        
        numbers = row[2]
        numbers = numbers.split(',') 

        # for each feature, we need to sum the skipping and inclusion counts 
        # 4 samples so 4 features
        if get_total_counts: 
            # get list of length 4  
            summations = summate_counts(numbers)
            
            assert len(summations)==4 and len(summated_sample_ordering)==4
            
            # assign each value in their respective order with correct sample naming
            for sample_name, value in zip(summated_sample_ordering, summations): 
                
                # include RBP, batch, and cell line info in the sample name 
                sample_name = "{}-{}".format(rbp_cell_line, sample_name)
                
                matrix_dict[feature][sample_name] = value    
                
            
        # OR ELSE for each feature, separate skipping and inculsion counts 
        # 4 samples * (SJC/IJC) = 8 features 
        elif not get_total_counts:
            
            assert len(numbers)==8 and len(non_summated_sample_ordering)==8
            
            # assign each value in correct order 
            for sample_name, value in zip(non_summated_sample_ordering, numbers): 
               
                # include RBP, batch, and cell line info in the sample name 
                sample_name = "{}-{}".format(rbp_cell_line, sample_name)
                
                matrix_dict[feature][sample_name] = value   
    
    # return dataframe where features are the index and columns are samples 
    return pd.DataFrame.from_dict(
        matrix_dict, 
        orient="index"
    )
    

def summate_counts(numbers): 
    """
    Parameters: 
        numbers (list): list of numbers to be summated in particular order
    
    Returns: 
        summation_list (list): list of numbers that have come as a result of correct summations
    
    """
        
    # list to save the summations 
    summation_list = []
    
    # which list indices should be pulled out for each sample
    # when summating and finding out counts per sample per feature 
    sample_count_summation_indices = {
        "KO_1": [0,2],
        "KO_2": [1,3],
        "CTRL_1": [4,6],
        "CTRL_2": [5,7],
    }
                
    # to get counts for each sample
    # pull the correct indices corresponding to junction and skipping counts for that sample 
    # sum that up and save those results in dictionary
    for sample in sample_count_summation_indices: 
        summation_list.append(
            sum(
                [int(numbers[index]) for index in sample_count_summation_indices[sample] ]
            )
        
        )
        
    return summation_list
        
        

## Validation of Matrix Creation Algorithm

In [4]:
random_file = '/scratch/jve4pt/ABCF1-BGHLV30-HepG2/MATS_output/A3SS.MATS.JunctionCountOnly_corrected_pval_yogi_february_2024.tsv'

original_df = pd.read_csv(random_file, sep="\t")

original_df.iloc[:,3:16].head()

,chr,strand,longExonStart_0base,longExonEnd,shortES,shortEE,flankingES,flankingEE,ID.1,IJC_SAMPLE_1,SJC_SAMPLE_1,IJC_SAMPLE_2,SJC_SAMPLE_2
0,chr17,+,7480886,7481024,7480940,7481024,7480661,7480805,7779,"5806,1983","241,0","1779,2855","2,0"
1,chr1,-,155638417,155638568,155638417,155638508,155640110,155640255,10717,"359,135","174,56","112,205","21,37"
2,chr6,-,31118231,31118342,31118231,31118333,31118501,31118637,17904,"159,57","0,0","29,44","4,7"
3,chr19,+,58904725,58904854,58904737,58904854,58904342,58904552,6652,"2427,760","75,2","496,952","0,0"
4,chr20,-,1426310,1426794,1426310,1426475,1433137,1433275,8471,"56,21","1176,348","22,52","194,373"


In [5]:
matrixize_rmats_table(file = random_file, get_total_counts=True).head()


,ABCF1-BGHLV30-HepG2-KO_Sample_1,ABCF1-BGHLV30-HepG2-KO_Sample_2,ABCF1-BGHLV30-HepG2-CTRL_Sample_1,ABCF1-BGHLV30-HepG2-CTRL_Sample_2
chr17_+_7480886_7481024_7480940_7481024_7480661_7480805,6047,1983,1781,2855
chr1_-_155638417_155638568_155638417_155638508_155640110_155640255,533,191,133,242
chr6_-_31118231_31118342_31118231_31118333_31118501_31118637,159,57,33,51
chr19_+_58904725_58904854_58904737_58904854_58904342_58904552,2502,762,496,952
chr20_-_1426310_1426794_1426310_1426475_1433137_1433275,1232,369,216,425


In [6]:
matrixize_rmats_table(file = random_file, get_total_counts=False).head()


,ABCF1-BGHLV30-HepG2-KO_Sample_1_IJC,ABCF1-BGHLV30-HepG2-KO_Sample_2_IJC,ABCF1-BGHLV30-HepG2-KO_Sample_1_SJC,ABCF1-BGHLV30-HepG2-KO_Sample_2_SJC,ABCF1-BGHLV30-HepG2-CTRL_Sample_1_IJC,ABCF1-BGHLV30-HepG2-CTRL_Sample_2_IJC,ABCF1-BGHLV30-HepG2-CTRL_Sample_1_SJC,ABCF1-BGHLV30-HepG2-CTRL_Sample_2_SJC
chr17_+_7480886_7481024_7480940_7481024_7480661_7480805,5806,1983,241,0,1779,2855,2,0
chr1_-_155638417_155638568_155638417_155638508_155640110_155640255,359,135,174,56,112,205,21,37
chr6_-_31118231_31118342_31118231_31118333_31118501_31118637,159,57,0,0,29,44,4,7
chr19_+_58904725_58904854_58904737_58904854_58904342_58904552,2427,760,75,2,496,952,0,0
chr20_-_1426310_1426794_1426310_1426475_1433137_1433275,56,21,1176,348,22,52,194,373


## Create All Matrices

In [7]:

# for each cell line 
for cell_line in cell_lines: 
    
    # for each splice type 
    for splice_type in splice_types: 
        
        # create output file name 
        if get_total_counts: 
            output_file = "../output/summated/{}_{}_summated.csv.gz".format(cell_line, splice_type)
        elif not get_total_counts: 
            output_file = "../output/non-summated/{}_{}_non-summated.csv.gz".format(cell_line, splice_type)
            
        # make sure that matrix doesn't already exist
        if os.path.isfile(output_file): 
            "Skipping {}".format(output_file)
            continue
        
        # dataframe for outer joining 
        join_df = pd.DataFrame()
        
        "{} {}".format(cell_line, splice_type)
        
        # get all files matching cell line and splice type 
        matching_files = sorted(glob.glob(
            "/scratch/jve4pt/*{}*/**/{}/{}*yogi*.tsv".format(cell_line, norm_or_not_pattern, splice_type), 
            recursive=True
        ))
        
        # for each rMATS table
        for file in matching_files: 
            
            # convert rMATS table to matrix of counts 
            matrix = matrixize_rmats_table(
                file = file, 
                get_total_counts = get_total_counts
            )
            
            if get_total_counts: 
                assert len(matrix.columns)==4
            elif not get_total_counts: 
                assert len(matrix.columns)==8
                
            join_df = join_df.join(matrix, how="outer")
        
        # check that number of samples is as expected
        if get_total_counts: 
            assert len(join_df.columns)==len(matching_files)*4
        elif not get_total_counts: 
            assert len(join_df.columns)==len(matching_files)*8
                    
        # summated data 
        if get_total_counts: 
            # save the matrices to disk
            join_df.to_csv(output_file)
        # not summated data 
        elif not get_total_counts: 
            # save the matrices to disk
            join_df.to_csv(output_file)


'HepG2 A3SS'

'HepG2 A5SS'

'HepG2 SE'

'HepG2 MXE'

'HepG2 RI'

'K562 A3SS'

'K562 A5SS'

'K562 SE'

'K562 MXE'

'K562 RI'